In [1]:
pip install langchain langchain-groq langchain-community chromadb sentence-transformers langchain_text_splitters


  Using cached langchain-1.3.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph-1.2.1-py3-none-any.whl.metadata (8.0 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.8.5-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

Load and Chunk the Document

In [2]:
# !pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Laksh.Sharma\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the file you just created
loader = TextLoader("../platform-knowledge.txt")
documents = loader.load()

# Split text into chunks so the AI can search effectively
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from your text file.")


Created 11 chunks from your text file.


C:\Users\Laksh.Sharma\AppData\Local\Temp\ipykernel_3856\1845438160.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embedding model (runs on your CPU)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the Vector Database and save it to a folder named 'db'
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory="../chroma_db"
)

print("Vector Database created and saved to ../chroma_db")


C:\Users\Laksh.Sharma\AppData\Local\Temp\ipykernel_3856\1484323832.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7780.38it/s]


Vector Database created and saved to ../chroma_db


In [5]:
query = "How can i view support contact?"
docs = vector_db.similarity_search(query, k=2)
# docs = [doc for doc, score in docs if score < 0.5]
context = "\n\n".join([doc.page_content for doc in docs])

from langchain_groq import ChatGroq
from app.core.config import GROQ_API_KEY
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.2   # ✅ more precise answers
)

prompt = f"""
You are an AI assistant for an Ed-Tech platform.

Use ONLY the context below to answer the question.

Rules:
- Keep answer short (max 2-3 lines)
- Be precise
- No extra information

Context:
{context}

Question: {query}

Answer:
"""

# ✅ Step 5: Generate answer
response = llm.invoke(prompt)

print(response.content)



ModuleNotFoundError: No module named 'app'